# MobileNetV3-Small — Leaf Classification

Pretrained MobileNetV3-Small fine-tuned on `data/train|val|test/` (grouped split, produced by
`01_preprocessing.ipynb`), with class-weighted loss from `data/class_weights.json`.

**This notebook is unexecuted — run all cells yourself to train.**


In [ ]:
import os, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

# ── Paths ──────────────────────────────────────
DATA_DIR   = Path("data")
OUTPUT_DIR = Path("mobilenetv3_small_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ─────────────────────────────
CLASSES     = sorted(d.name for d in (DATA_DIR / "train").iterdir() if d.is_dir())
NUM_CLASSES = len(CLASSES)
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 50
PATIENCE    = 5
MIN_EPOCHS_BEFORE_STOP = 15
LR          = 1e-4       # lower than the from-scratch CNN's 1e-3 — this is fine-tuning pretrained weights
SEED        = 42
NUM_WORKERS = 2

# Fixed categorical color per class (never cycled), reused across every chart in this notebook
CLASS_COLORS = dict(zip(CLASSES, ["#2a78d6", "#008300", "#e87ba4", "#eda100",
                                   "#1baf7a", "#eb6834", "#4a3aa7", "#e34948"]))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. Check nvidia-smi, then re-run.")

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

print(f"Classes ({NUM_CLASSES}): {CLASSES}")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Data pipeline

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=25, border_mode=0, fill=0, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


class LeafDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None):
        self.transform = transform
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.samples = []
        for c in classes:
            for p in sorted(Path(root_dir, c).glob("*.jpg")):
                self.samples.append((str(p), self.class_to_idx[c]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, label


train_ds = LeafDataset(DATA_DIR / "train", CLASSES, transform=train_transform)
val_ds   = LeafDataset(DATA_DIR / "val",   CLASSES, transform=eval_transform)
test_ds  = LeafDataset(DATA_DIR / "test",  CLASSES, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")


## Class weights\nLoaded from `data/class_weights.json` (computed on the train split only).

In [ ]:
with open(DATA_DIR / "class_weights.json") as f:
    class_weights = json.load(f)

class_weights_tensor = torch.tensor([class_weights[c] for c in CLASSES], dtype=torch.float32).to(DEVICE)
for c, w in class_weights.items():
    print(f"  {c:12s}: {w:.3f}")


## Model: MobileNetV3-Small (ImageNet pretrained)\nBackbone from `torchvision.models.mobilenet_v3_small`, classifier head replaced for 8 classes, fully fine-tuned. *(Assuming the Small variant per your first message — swap to `mobilenet_v3_large` / `MobileNet_V3_Large_Weights` in the model cell if you meant Large.)*

In [ ]:
def replace_classifier(base_model, attr_name, num_classes):
    """Swap the final Linear layer of a torchvision classifier head, keeping in_features."""
    module = getattr(base_model, attr_name)
    if isinstance(module, nn.Linear):
        setattr(base_model, attr_name, nn.Linear(module.in_features, num_classes))
    elif isinstance(module, nn.Sequential):
        for idx in reversed(range(len(module))):
            if isinstance(module[idx], nn.Linear):
                module[idx] = nn.Linear(module[idx].in_features, num_classes)
                break
    else:
        raise TypeError(f"Unexpected classifier type: {type(module)}")
    return base_model


def build_model(num_classes):
    m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    return replace_classifier(m, "classifier", num_classes)

model     = build_model(NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MobileNetV3-Small — trainable parameters: {total_params:,}")


## Training (AMP + early stopping, best model saved on val loss)

In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / "best_mobilenetv3_small.pth"

scaler_amp        = torch.amp.GradScaler("cuda")
best_val_loss     = float("inf")
epochs_no_improve = 0
train_loss_hist, val_loss_hist = [], []
train_acc_hist,  val_acc_hist  = [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    run_loss, run_correct, run_total = 0.0, 0, 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False):
        inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()

        with torch.no_grad():
            preds = outputs.argmax(1)

        run_loss    += loss.item() * inputs.size(0)
        run_correct += (preds == labels).sum().item()
        run_total   += inputs.size(0)

    train_loss = run_loss / run_total
    train_acc  = run_correct / run_total
    train_loss_hist.append(train_loss)
    train_acc_hist.append(train_acc)

    # ── Validate ──
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False):
            inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda"):
                outputs = model(inputs)
                loss    = criterion(outputs, labels)
            val_loss    += loss.item() * inputs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += inputs.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total
    val_loss_hist.append(val_loss)
    val_acc_hist.append(val_acc)

    print(f"[Epoch {epoch:02d}]  "
          f"Train Acc: {train_acc:.4f}  Val Acc: {val_acc:.4f}  "
          f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")

    # ── Early stopping ──
    if val_loss < best_val_loss:
        best_val_loss     = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> Best model saved  (val_loss={best_val_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epoch < MIN_EPOCHS_BEFORE_STOP:
            print(f"  -> No improvement for {epochs_no_improve}/{PATIENCE}, "
                  f"but continuing until at least epoch {MIN_EPOCHS_BEFORE_STOP}.")
        elif epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} after {epochs_no_improve} epochs without improvement.")
            break

print(f"\nTraining complete. Best model -> {BEST_MODEL_PATH}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_acc_hist, label="Train", color="#2a78d6")
axes[0].plot(val_acc_hist,   label="Val",   color="#008300")
axes[0].set_title("Accuracy — MobileNetV3-Small"); axes[0].set_xlabel("Epoch"); axes[0].legend(frameon=False)
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].plot(train_loss_hist, label="Train", color="#2a78d6")
axes[1].plot(val_loss_hist,   label="Val",   color="#008300")
axes[1].set_title("Loss — MobileNetV3-Small"); axes[1].set_xlabel("Epoch"); axes[1].legend(frameon=False)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()


## Test evaluation\nLoad the best (lowest val-loss) checkpoint and evaluate once, on the held-out test split.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Inference", leave=False):
        inputs = inputs.to(DEVICE)
        with torch.amp.autocast("cuda"):
            outputs = model(inputs)
        probs = F.softmax(outputs, dim=1).float().cpu().numpy()
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)
test_acc   = (all_preds == all_labels).mean()
print(f"Test Accuracy (MobileNetV3-Small): {test_acc * 100:.2f}%")
print(f"Correct: {(all_preds == all_labels).sum()} / {len(all_labels)}")


### Confusion matrix & classification report

In [ ]:
cm   = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(cmap="Blues", values_format="d")
plt.title("Test Confusion Matrix — MobileNetV3-Small")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

report_txt = classification_report(all_labels, all_preds, target_names=CLASSES, digits=4)
print(report_txt)
with open(OUTPUT_DIR / "classification_report.txt", "w") as f:
    f.write(report_txt)
pd.DataFrame(
    classification_report(all_labels, all_preds, target_names=CLASSES, output_dict=True, digits=4)
).transpose().to_csv(OUTPUT_DIR / "classification_report.csv")


### Per-class precision / recall / F1

In [ ]:
precision = precision_score(all_labels, all_preds, average=None, labels=range(NUM_CLASSES))
recall    = recall_score(all_labels,    all_preds, average=None, labels=range(NUM_CLASSES))
f1        = f1_score(all_labels,        all_preds, average=None, labels=range(NUM_CLASSES))
x = np.arange(NUM_CLASSES)
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, precision, w, label="Precision", color="#2a78d6")
ax.bar(x,     recall,    w, label="Recall",    color="#008300")
ax.bar(x + w, f1,        w, label="F1",        color="#e87ba4")
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
ax.set_ylim(0, 1.1)
ax.set_title("Per-Class Metrics — MobileNetV3-Small (Test Set)")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5, color="#dddddd", zorder=0)
ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_class_metrics.png", dpi=150)
plt.show()

pd.DataFrame({
    "Sample":     range(len(all_labels)),
    "True Label": [CLASSES[i] for i in all_labels],
    "Pred Label": [CLASSES[i] for i in all_preds],
    "Correct":    (all_preds == all_labels).astype(int)
}).to_csv(OUTPUT_DIR / "predictions.csv", index=False)


### ROC curve (one-vs-rest)

In [ ]:
y_true_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), all_probs.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

fig, ax = plt.subplots(figsize=(8, 7))
for i, cls in enumerate(CLASSES):
    ax.plot(fpr[i], tpr[i], color=CLASS_COLORS[cls], lw=2, label=f"{cls} (AUC={roc_auc[i]:.3f})")
ax.plot(fpr["micro"], tpr["micro"], color="black", lw=2, linestyle="--",
        label=f"micro-average (AUC={roc_auc['micro']:.3f})")
ax.plot([0, 1], [0, 1], color="#999999", lw=1, linestyle=":")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (One-vs-Rest) — MobileNetV3-Small")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve.png", dpi=150)
plt.show()

print("Per-class AUC:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:12s}: {roc_auc[i]:.4f}")
print(f"  {'micro-avg':12s}: {roc_auc['micro']:.4f}")


### Precision-Recall curve

In [ ]:
precision_d, recall_d, ap = {}, {}, {}
for i in range(NUM_CLASSES):
    precision_d[i], recall_d[i], _ = precision_recall_curve(y_true_bin[:, i], all_probs[:, i])
    ap[i] = average_precision_score(y_true_bin[:, i], all_probs[:, i])

fig, ax = plt.subplots(figsize=(8, 7))
for i, cls in enumerate(CLASSES):
    ax.plot(recall_d[i], precision_d[i], color=CLASS_COLORS[cls], lw=2, label=f"{cls} (AP={ap[i]:.3f})")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — MobileNetV3-Small")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower left", fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pr_curve.png", dpi=150)
plt.show()

print("Per-class Average Precision:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:12s}: {ap[i]:.4f}")


## Feature / embedding extraction\nPenultimate-layer feature vectors (the input to the final classification layer), saved per split for a future mRMR + SVM / hybrid ML stage. Uses a forward hook so it works generically regardless of the classifier head's exact shape.

In [ ]:
def get_final_linear(module_holder, attr_name="classifier"):
    """Return the actual nn.Linear submodule that produces the class logits,
    whether `classifier` is a bare Linear or a Sequential ending in one."""
    module = getattr(module_holder, attr_name)
    if isinstance(module, nn.Linear):
        return module
    elif isinstance(module, nn.Sequential):
        for layer in reversed(module):
            if isinstance(layer, nn.Linear):
                return layer
    raise TypeError(f"No Linear layer found in {attr_name}")


final_linear   = get_final_linear(model, "classifier")
embedding_dim  = final_linear.in_features
print(f"Embedding dimension (input to final Linear): {embedding_dim}")

_captured = {}

def _capture_hook(module, inputs, output):
    _captured["embedding"] = inputs[0].detach()

hook_handle = final_linear.register_forward_hook(_capture_hook)


@torch.no_grad()
def extract_embeddings(loader):
    model.eval()
    embs, labels = [], []
    for inputs, lbls in tqdm(loader, desc="Extracting embeddings", leave=False):
        inputs = inputs.to(DEVICE)
        with torch.amp.autocast("cuda"):
            _ = model(inputs)
        embs.append(_captured["embedding"].float().cpu().numpy())
        labels.append(lbls.numpy())
    return np.concatenate(embs), np.concatenate(labels)


# Deterministic (non-augmented) view of the train split, so features are stable/reproducible
# — unlike train_loader, which applies random augmentation for the training step above.
train_eval_ds      = LeafDataset(DATA_DIR / "train", CLASSES, transform=eval_transform)
train_embed_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)

train_embs, train_y = extract_embeddings(train_embed_loader)
val_embs,   val_y   = extract_embeddings(val_loader)
test_embs,  test_y  = extract_embeddings(test_loader)

hook_handle.remove()

np.save(OUTPUT_DIR / "train_embs.npy", train_embs)
np.save(OUTPUT_DIR / "val_embs.npy",   val_embs)
np.save(OUTPUT_DIR / "test_embs.npy",  test_embs)
np.save(OUTPUT_DIR / "train_y.npy",    train_y)
np.save(OUTPUT_DIR / "val_y.npy",      val_y)
np.save(OUTPUT_DIR / "test_y.npy",     test_y)

print(f"train_embs: {train_embs.shape}  val_embs: {val_embs.shape}  test_embs: {test_embs.shape}")
print(f"Saved to {OUTPUT_DIR.resolve()}")


## Summary

Artifacts saved to `mobilenetv3_small_results/`:
- `best_mobilenetv3_small.pth` — best checkpoint (lowest val loss)
- `training_curves.png` — train/val accuracy & loss over epochs
- `confusion_matrix.png`, `classification_report.{txt,csv}`, `per_class_metrics.png`, `predictions.csv`
- `roc_curve.png` — one-vs-rest ROC with per-class + micro-average AUC
- `pr_curve.png` — per-class precision-recall curves with average precision
- `train_embs.npy` / `val_embs.npy` / `test_embs.npy` (+ matching `*_y.npy` labels) — penultimate-layer
  features per split (dimension printed at extraction time), for a future mRMR + SVM (or other ML) hybrid
  stage, same pattern as the FYP CNN6+mRMR+SVM pipeline
